# Stage 2: Face CNN + localized Chebyshev GCN

Follow `notebooks/06_Graph_Convnets/code01_solution.ipynb`, especially `graph_conv_cheby`: T₀X = X, T₁X = L̃X, TₖX = 2L̃Tₖ₋₁X − Tₖ₋₂X; concatenate terms and apply a learned Linear. This adapts the tutorial's graph-level MNIST network to **per-face classification**. No graph coarsening, pooling, or node-order flattening: each face keeps its output.

Use **one graph convolution**, ReLU, and a pointwise two-layer MLP. Two-hop and three-hop variants have polynomial degree 2 and 3 respectively (the tutorial's term count K is 3 and 4). One graph layer keeps the total graph receptive field at the named hop count. The CNN and pointwise MLP do not expand it. Edge geometry and shallow topology descriptors are not used.

Each solid uses its own simple undirected adjacency (remove self-loops, collapse parallel edges). L = I − D⁻¹ᐟ²AD⁻¹ᐟ². Scale using the normalized-Laplacian upper bound λmax=2, so L̃ = L − I = −D⁻¹ᐟ²AD⁻¹ᐟ². No eigenvectors or eigenvalue computation are needed. Isolated nodes use L=I, L̃=0, with self information retained through T₀. Batches use a sparse block diagonal operator, preventing messages between solids.

Reference: [Defferrard et al., ChebNet](https://arxiv.org/abs/1606.09375). This is the polynomial GCN stage, rather than UV-Net's edge-conditioned graph network.


In [ ]:
import os
os.environ["DGLBACKEND"] = "pytorch"

import csv
import hashlib
import json
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from uvnet_project.checkpointing import seed_everything, save_training_checkpoint, load_training_checkpoint
from uvnet_project.face_cnn import FaceCNN
from uvnet_project.chebyshev import scaled_laplacian, collate_cheb, ChebFaceClassifier
from uvnet_project.load_mfcad import DATA_DIR, _load_sample, _load_splits
from uvnet_project.evaluation import evaluate


## Reproducibility and resume

Fresh runs reset Python, NumPy and Torch CPU/CUDA seeds. Each epoch exports weights and a full atomic checkpoint with optimizer, AMP scaler when enabled, initial seed/RNG states, histories and settings. Resume a completed epoch with `resume_checkpoint` and set `num_epochs` to the total target. Keep the same input caches, dataset order, software and GPU. Do not reset the seed inside the epoch loop. Mid-epoch interruption restarts from the last completed epoch.

Use a fresh kernel for deterministic CUDA settings. Sparse Chebyshev arithmetic stays float32; joint CNN and pointwise layers use AMP on CUDA. No training is run during notebook preparation.


## Settings and baseline CNN

Run `cheb_hops=2` and `3`. Use `training_mode="frozen"` to cache baseline CNN embeddings, or `"joint"` to update the CNN too. Both start from the same baseline checkpoint. The graph width defaults to 64; embedding and MLP widths come from the baseline config. Each hop count/mode gets its own results folder. Comparisons contain baseline + Chebyshev variants only. Test data is not loaded. Matching run files are overwritten.


In [ ]:
seed = 0
deterministic = True
# Set a completed Chebyshev checkpoint to resume, then rerun preparation cells.
# resume_checkpoint = DATA_DIR.parent.parent / "results/cheb2_frozen_run_01/epoch_010/checkpoint.pt"
resume_checkpoint = None
seed_everything(seed, deterministic=deterministic)

project_dir = DATA_DIR.parent.parent
baseline_checkpoint = project_dir / "results/baseline_run_01/epoch_020/checkpoint.pt"
training_mode = "frozen"  # "frozen" or "joint"
cheb_hops = 2  # 2 or 3; polynomial degree, NOT number of terms
graph_dim = 64  # one graph convolution, 64 -> graph_dim
num_epochs = 20
batch_size = 128  # solids per batch
learning_rate = 0.001
run_dir = project_dir / "results" / f"cheb{cheb_hops}_{training_mode}_run_01"
cache_dir = project_dir / "data/feature_cache"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if training_mode not in ("frozen", "joint") or cheb_hops not in (2, 3):
    raise ValueError("Choose frozen/joint and cheb_hops=2 or 3.")
if min(num_epochs, batch_size) < 1:
    raise ValueError("Epoch count and batch size must be positive.")
if not baseline_checkpoint.is_file():
    raise FileNotFoundError(f"Copy the completed baseline checkpoint here first: {baseline_checkpoint}")
checkpoint = torch.load(baseline_checkpoint, map_location="cpu", weights_only=True)
config = checkpoint["config"]
embedding_dim, hidden_dim = config["embedding_dim"], config["hidden_dim"]
num_classes, in_channels = config["num_classes"], config["in_channels"]
baseline_hash = hashlib.sha256(baseline_checkpoint.read_bytes()).hexdigest()
cnn_state = {key.removeprefix("0."): value for key, value in checkpoint["model_state_dict"].items()
             if key.startswith("0.")}
face_cnn = FaceCNN(embedding_dim=embedding_dim, in_channels=in_channels)
face_cnn.load_state_dict(cnn_state, strict=True)
face_cnn.requires_grad_(False)
face_cnn = face_cnn.to(device).eval()
print("Device:", device, "Mode:", training_mode, "Chebyshev hops:", cheb_hops)


## Cache sparse graph operators and CNN inputs

Cache one sparse scaled Laplacian per solid and the aligned face labels. Frozen mode encodes the baseline geometry once; joint mode stores the raw UV grids. The operator is independent of CNN weights and hop count; both hop variants reuse the same mode-specific input cache. Metadata records dataset IDs, graph policy and baseline hash. Retain these caches when resuming. Delete caches or increment `cache_version` if dataset contents or preprocessing change.


In [ ]:
def metadata_key(metadata):
    return hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:16]


@torch.no_grad()
def prepare_inputs(split, encode_batch_size=128):
    sample_ids = _load_splits()[split]
    metadata = dict(cache_version=1, baseline_hash=baseline_hash, training_mode=training_mode,
                    sample_ids=sample_ids, embedding_dim=embedding_dim, in_channels=in_channels,
                    graph_policy="simple_undirected_no_loops_normalized_lmax2_isolated_L_identity")
    path = cache_dir / f"cheb_inputs_{split}_{metadata_key(metadata)}.pt"
    if path.is_file():
        cached = torch.load(path, map_location="cpu", weights_only=True)
        if cached["metadata"] != metadata:
            raise ValueError("Chebyshev input-cache metadata mismatch.")
        print("Using input cache:", path)
        return cached["samples"], sample_ids
    if resume_checkpoint is not None:
        raise FileNotFoundError(f"Retain the original Chebyshev input cache for resume: {path}")
    face_cnn.eval()
    samples = []
    for start in tqdm(range(0, len(sample_ids), encode_batch_size), desc=f"Cache Chebyshev inputs {split}"):
        solids = [_load_sample(sample_id) for sample_id in sample_ids[start:start + encode_batch_size]]
        embeddings = None
        if training_mode == "frozen":
            sizes = [solid["graph"].num_nodes() for solid in solids]
            grids = torch.cat([solid["graph"].ndata["x"] for solid in solids])
            grids = grids.permute(0, 3, 1, 2).contiguous().float().to(device)
            embeddings = face_cnn(grids).cpu().split(sizes)
        for offset, solid in enumerate(solids):
            laplacian = scaled_laplacian(solid["graph"])
            labels = torch.tensor([face["segment"]["index"] for face in solid["faces"]], dtype=torch.long)
            geometry = (embeddings[offset] if training_mode == "frozen" else
                        solid["graph"].ndata["x"].permute(0, 3, 1, 2).contiguous().float())
            samples.append((geometry, laplacian, labels))
    cache_dir.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".pt.tmp")
    torch.save(dict(metadata=metadata, samples=samples), temporary)
    os.replace(temporary, path)
    return samples, sample_ids


train_samples, train_ids = prepare_inputs("train")
validation_samples, validation_ids = prepare_inputs("validation")


## Build the Chebyshev face classifier

`Face CNN (64D) -> ChebyshevConv (graph_dim) -> ReLU -> Linear (hidden_dim) -> ReLU -> Linear (16 logits)`.

Frozen mode trains the graph convolution and MLP. Joint mode also trains the CNN and its batch-normalization statistics. Both modes use the same initialized graph/head weights. Chebyshev terms are computed by sparse recurrence following the tutorial; no dense NxN matrix or eigendecomposition is used. The zero-order term supplies each face's own features without adding self-loops.


In [ ]:
def build_model():
    head = ChebFaceClassifier(embedding_dim, graph_dim, hidden_dim, cheb_hops, num_classes)
    if training_mode == "joint":
        # Keep initialization of classifier/shuffling identical between modes.
        with torch.random.fork_rng(devices=[]):
            encoder = FaceCNN(embedding_dim=embedding_dim, in_channels=in_channels)
        encoder.load_state_dict(cnn_state, strict=True)
        encoder.requires_grad_(True)
        head.face_cnn = encoder
    return head.to(device)


seed_everything(seed, deterministic=deterministic)
model = build_model()
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## Batch solids without connecting them

Use a sparse block-diagonal Laplacian with concatenated face features and labels. `batch_size` counts solids, not faces. Face order is preserved within each solid. Check the output shape and finite values before training.


In [ ]:
loader_options = dict(batch_size=batch_size, collate_fn=collate_cheb,
                      num_workers=0, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_samples, shuffle=True, **loader_options)
train_evaluation_loader = DataLoader(train_samples, shuffle=False, **loader_options)
validation_loader = DataLoader(validation_samples, shuffle=False, **loader_options)
inputs, labels = next(iter(validation_loader))
model.eval()
with torch.no_grad():
    logits = model(inputs.to(device))
assert logits.shape == (len(labels), num_classes)
assert torch.isfinite(logits).all()
print("Logits:", tuple(logits.shape), "Sparse entries:", inputs.laplacian._nnz())


## Train the chosen Chebyshev variant — run when ready

Record batch training loss, evaluate validation each epoch, and save weights plus resumable checkpoints every epoch. Frozen mode uses float32; joint CUDA mode uses AMP with saved scaler state. Training metrics accumulate during updates; validation uses evaluation mode. Prediction probabilities are saved only at the final epoch. Loss plotting is separate and runs after training finishes.


In [ ]:
seed_everything(seed, deterministic=deterministic)
if resume_checkpoint is None:
    model = build_model()
use_amp = training_mode == "joint" and device.type == "cuda"
if use_amp:
    model = model.to(memory_format=torch.channels_last)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
loss_fn = nn.CrossEntropyLoss()
history, batch_history = [], []
global_step = 0

run_config = dict(stage="chebyshev", seed=seed, deterministic=deterministic,
                  embedding_dim=embedding_dim, cheb_hops=cheb_hops, graph_dim=graph_dim,
                  num_graph_layers=1, graph_policy="simple_undirected_no_loops_normalized_lmax2_isolated_L_identity",
                  hidden_dim=hidden_dim, num_classes=num_classes,
                  learning_rate=learning_rate, batch_size=batch_size,
                  baseline_hash=baseline_hash, train_sample_ids=train_ids,
                  validation_sample_ids=validation_ids, training_mode=training_mode,
                  mixed_precision=use_amp)
start_epoch = 1
if resume_checkpoint is not None:
    resume_info = torch.load(resume_checkpoint, map_location="cpu", weights_only=True)
    saved_config = resume_info["config"]
    # Resume must use the same hop count, graph settings and training mode.
    if saved_config.get("training_mode", "frozen") != training_mode:
        raise ValueError("Cannot resume a different training mode.")
    if saved_config.get("mixed_precision", False) != use_amp:
        raise ValueError("Resume mixed-precision setting does not match.")
    expected_config = {key: value for key, value in run_config.items()
                       if key not in ("training_mode", "mixed_precision") or key in saved_config}
    restored = load_training_checkpoint(
        resume_checkpoint, model, optimizer, expected_config=expected_config, scaler=scaler if use_amp else None,
    )
    start_epoch = restored["epoch"] + 1
    history = restored["history"]
    batch_history = restored["batch_history"]
    global_step = batch_history[-1]["step"] if batch_history else 0
    if num_epochs < start_epoch:
        raise ValueError("Set num_epochs above the completed checkpoint epoch to continue training.")
    print(f"Resuming at epoch {start_epoch}; target is epoch {num_epochs}.")
run_dir.mkdir(parents=True, exist_ok=True)


@torch.no_grad()
def validation_metrics():
    model.eval()
    loss_total = torch.zeros((), device=device)
    correct_total = torch.zeros((), dtype=torch.long, device=device)
    faces = 0
    for features, labels in validation_loader:
        features, labels = features.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(features)
            loss = loss_fn(logits, labels)
        loss_total += loss * labels.numel()
        correct_total += (logits.argmax(1) == labels).sum()
        faces += labels.numel()
    return loss_total.item() / faces, correct_total.item() / faces


for epoch in tqdm(range(start_epoch, num_epochs + 1), desc="Epochs"):
    model.train()
    losses, correct, sizes = [], [], []
    for features, labels in tqdm(train_loader, desc=f"Epoch {epoch}: training", leave=False):
        features, labels = features.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(features)
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.detach())
        correct.append((logits.detach().argmax(1) == labels).sum())
        sizes.append(labels.numel())

    loss_values = torch.stack(losses).cpu().tolist()
    train_loss = sum(loss * size for loss, size in zip(loss_values, sizes)) / sum(sizes)
    train_accuracy = torch.stack(correct).sum().item() / sum(sizes)
    for batch_index, (batch_loss, count) in enumerate(zip(loss_values, sizes), 1):
        global_step += 1
        batch_history.append(dict(step=global_step, epoch=epoch, batch=batch_index,
                                  loss=batch_loss, faces=count))
    val_loss, val_accuracy = validation_metrics()
    row = dict(epoch=epoch, step=global_step, train_loss=train_loss,
               train_accuracy=train_accuracy, validation_loss=val_loss,
               validation_accuracy=val_accuracy)
    history.append(row)
    checkpoint_dir = run_dir / f"epoch_{epoch:03d}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    save_training_checkpoint(
        checkpoint_dir / "checkpoint.pt", model, optimizer, epoch=epoch, seed=seed,
        config={**run_config, "num_epochs": num_epochs, "checkpoint_every": 1},
        history=history, batch_history=batch_history, scaler=scaler if use_amp else None,
        extra=dict( baseline_checkpoint=str(baseline_checkpoint), baseline_hash=baseline_hash, face_cnn_state_dict=(model.face_cnn if training_mode == "joint" else face_cnn).state_dict(), train_sample_ids=train_ids, validation_sample_ids=validation_ids),
    )
    with (run_dir / "history.csv").open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(row))
        writer.writeheader()
        writer.writerows(history)
    tqdm.write(f"Epoch {epoch:02d}/{num_epochs} | train loss {train_loss:.4f}, "
               f"accuracy {train_accuracy:.2%} | validation loss {val_loss:.4f}, "
               f"accuracy {val_accuracy:.2%}")
    with (run_dir / "batch_loss.csv").open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(batch_history[0]))
        writer.writeheader()
        writer.writerows(batch_history)

final_train_results = evaluate(model, train_evaluation_loader,
                               checkpoint_dir / "train_predictions.pt", device=device)
final_validation_results = evaluate(model, validation_loader,
                                    checkpoint_dir / "validation_predictions.pt", device=device)
print("Final predictions saved to:", checkpoint_dir)


## Loss plot after training
Run after the training cell completes.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

batch_log = pd.DataFrame(batch_history)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_log["step"], batch_log["loss"], alpha=0.25, label="Training batch loss")
ax.plot(batch_log["step"], batch_log["loss"].rolling(20, min_periods=1).mean(),
        label="Training loss (20-batch moving average)")
ax.plot([row["step"] for row in history],
        [row["validation_loss"] for row in history], marker="o", label="Validation loss")
ax.set_xlabel("Training step (batch)")
ax.set_ylabel("Mean cross-entropy loss per face")
ax.set_title(f"{training_mode.capitalize()} CNN + Chebyshev ({cheb_hops} hops)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(run_dir / "loss.png", dpi=150)
plt.show()


## Accuracy and IoU: baseline + Chebyshev variants

Read final cached training/validation predictions for the baseline and completed Chebyshev runs only. Show checkpoint epoch, loss, accuracy, unweighted mean IoU (including stock), and per-class IoU. Missing runs are skipped. No inference or training.


In [ ]:
# Compare cached final predictions across all completed variants.
from pathlib import Path
import json
import pandas as pd
import torch
from IPython.display import display
from uvnet_project.load_mfcad import DATA_DIR

project_dir = DATA_DIR.parent.parent
import re
variant_dirs = {"Geometry only": project_dir / "results/baseline_run_01"}
for folder in sorted((project_dir / "results").glob("cheb*_run_*")):
    match = re.fullmatch(r"cheb(\d+)_(frozen|joint)_run_(\d+)", folder.name)
    if match:
        hops, mode, run_id = match.groups()
        label = f"{mode.capitalize()} CNN + Chebyshev ({hops} hops, run {run_id})"
        variant_dirs[label] = folder
class_names = {}
for label_file in (DATA_DIR / "labels").glob("*.json"):
    with label_file.open() as file:
        faces = json.load(file)["body"]["faces"]
    for face in faces:
        segment = face["segment"]
        class_names[segment["index"]] = segment["name"]
    if len(class_names) == 16:
        break

summary_rows, class_rows = [], []
for variant, directory in variant_dirs.items():
    # Final predictions are written only after training. Use the highest epoch
    # with both splits available, not a partially written final evaluation.
    completed = [folder for folder in directory.glob("epoch_*")
                 if folder.name.removeprefix("epoch_").isdigit()
                 and all((folder / f"{split}_predictions.pt").is_file()
                         for split in ("train", "validation"))]
    if not completed:
        print(f"No completed prediction files yet: {variant}")
        continue
    checkpoint_dir = max(completed, key=lambda folder: int(folder.name.split("_")[-1]))
    epoch = int(checkpoint_dir.name.split("_")[-1])
    for split in ("train", "validation"):
        results = torch.load(checkpoint_dir / f"{split}_predictions.pt",
                             map_location="cpu", weights_only=True)
        labels, predictions = results["labels"].long(), results["predictions"].long()
        num_classes = results["probabilities"].shape[1]
        confusion = torch.bincount(labels * num_classes + predictions,
                                  minlength=num_classes**2).reshape(num_classes, num_classes)
        intersection = confusion.diag()
        support = confusion.sum(1)
        union = support + confusion.sum(0) - intersection
        iou = intersection.double() / union.double().masked_fill(union == 0, float("nan"))
        per_class_accuracy = intersection.double() / support.double().masked_fill(support == 0, float("nan"))
        summary_rows.append(dict(variant=variant, epoch=epoch, split=split,
                                 faces=labels.numel(), loss=results["loss"],
                                 accuracy=intersection.sum().item() / labels.numel(),
                                 mean_iou=iou.nanmean().item()))
        for index in range(num_classes):
            class_rows.append(dict(variant=variant, epoch=epoch, split=split,
                                   class_index=index, class_name=class_names.get(index, str(index)),
                                   support=support[index].item(), iou=iou[index].item(),
                                   accuracy=per_class_accuracy[index].item()))

if summary_rows:
    summary = pd.DataFrame(summary_rows).set_index(["variant", "epoch", "split"])
    per_class_metrics = pd.DataFrame(class_rows)
    display(summary.style.format({"loss": "{:.4f}", "accuracy": "{:.2%}", "mean_iou": "{:.2%}"}))
    print("Per-class IoU (unweighted class mean above, including stock):")
    display(per_class_metrics.pivot(index=["class_index", "class_name"],
                                    columns=["variant", "split"], values="iou").style.format("{:.2%}"))
    summary.to_csv(project_dir / "results/cheb_variant_metrics.csv")
    per_class_metrics.to_csv(project_dir / "results/cheb_variant_per_class_metrics.csv", index=False)
else:
    print("Run training through final prediction export before comparing metrics.")


## Loss comparison: baseline + Chebyshev variants

Plot saved epoch-level training and validation losses for baseline and Chebyshev runs only. Two-hop/three-hop filters and frozen/joint modes remain separate. Other themes are excluded.


In [ ]:
# Comparison from saved histories; no inference or training.
import matplotlib.pyplot as plt
import pandas as pd

import re
variants = {"Geometry only": project_dir / "results/baseline_run_01/history.csv"}
for folder in sorted((project_dir / "results").glob("cheb*_run_*")):
    match = re.fullmatch(r"cheb(\d+)_(frozen|joint)_run_(\d+)", folder.name)
    if match:
        hops, mode, run_id = match.groups()
        label = f"{mode.capitalize()} CNN + Chebyshev ({hops} hops, run {run_id})"
        variants[label] = folder / "history.csv"
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plotted = 0
for label, path in variants.items():
    if not path.is_file():
        print(f"Not run yet: {label}")
        continue
    frame = pd.read_csv(path)
    if frame.empty:
        continue
    for ax, metric in zip(axes, ("train_loss", "validation_loss")):
        ax.plot(frame["epoch"], frame[metric], marker="o", markersize=3, label=label)
    plotted += 1
for ax, title in zip(axes, ("Training loss", "Validation loss")):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Mean cross-entropy per face")
    ax.grid(alpha=0.3)
    if plotted:
        ax.legend()
fig.tight_layout()
if plotted:
    comparison_path = project_dir / "results/cheb_variants_loss.png"
    fig.savefig(comparison_path, dpi=150)
    print("Saved comparison:", comparison_path)
plt.show()
